
 criteria using keyword matching and Word2Vec with weighted scoring.

In [4]:
# Step 1: Install required libraries (only needed once)
!pip install gensim scikit-learn pandas --quiet

# Step 2: Imports
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from collections import Counter

# Step 3: 
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']

suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "Strong in pricing and lead time but lacks security protocols. Focuses on sustainability.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing. Low-cost pricing with excellent lead time.",
    'Supplier4': "Very costly. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "Good security and compliance, decent delivery, quality-focused. Balanced approach with pricing.",
    'Supplier7': "Imbalanced approach with pricing. Good security. Bad lead time.",
    'Supplier8': "Excellent lead time and low pricing. Lacks sustainability credentials and weak compliance.",
    'Supplier9': "ISO certified, strong compliance, and sustainable processes. Moderate pricing and solid quality.",
    'Supplier10': "High-end quality assurance, strong security, but high cost and slower delivery times."
}

# Step 4: Extract keywords
keywords = nike_criteria + ['compliance', 'green', 'logistics']
def extract_keywords(text):
    return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]

tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]

# Step 5: Sentiment-based weighting
positive_words = {'good', 'great', 'excellent', 'strong', 'fast', 'high', 'balanced'}
negative_words = {'bad', 'poor', 'slow', 'weak', 'low', 'imbalanced', 'excessive', 'lacks', 'decreased'}

def get_sentiment_scale(word, full_text):
    for pos in positive_words:
        if f"{pos} {word}" in full_text:
            return 1.2
    for neg in negative_words:
        if f"{neg} {word}" in full_text:
            return -0.8
    return 0.9  # neutral

# Step 6: Train Word2Vec
model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1)

# Step 7: Scoring loop using all available words
results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    full_text = suppliers[name].lower()
    scores = []
    
    for crit in nike_criteria:
        if crit not in model.wv:
            scores.append(0.0)
            continue

        crit_vec = model.wv[crit].reshape(1, -1)

        # Use all available matching words (not just exact match)
        weighted_vectors = []
        for w in words:
            if w in model.wv:
                weight = get_sentiment_scale(w, full_text)
                weighted_vectors.append(model.wv[w] * word_freq[w] * weight)

        if weighted_vectors:
            avg_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0

        scores.append(sim)

    results_weighted[name] = scores

# Step 8: Final DataFrame
df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores["TotalMatchScore"] = df_scores.sum(axis=1)
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)
# Step 8: Final DataFrame
df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores["TotalMatchScore"] = df_scores.sum(axis=1)
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)
 
# Define Criteria Weights
criteria_weights = {
    'quality': 0.7,        # Most important
    'delivery': 0.25,
    'lead': 0.2,
    'pricing': 0.15,
    'security': 0.07,
    'sustainability': 0.03  # Least important
}
 
# Apply Weights to Final Scoring
df_scores["TotalMatchScore"] = sum(
    df_scores[crit] * criteria_weights[crit] for crit in nike_criteria
)
 
# Sort Suppliers Based on Weighted Scores// recoreded for paper
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)
# Step 9: Show result
df_scores


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier10,0.514187,0.616093,0.716073,0.034708,0.073316,-0.297325,0.620887
Supplier1,0.713864,0.511989,-0.018505,-0.063470,-0.015921,0.290926,0.531586
Supplier3,0.026963,0.508563,0.093771,0.440679,0.456907,-0.162680,0.521902
Supplier9,0.198926,0.411566,0.225424,0.462071,0.145075,-0.148903,0.447466
Supplier4,0.018277,0.124863,0.011072,0.042373,1.000000,-0.174248,0.293877
Supplier6,0.486367,0.061025,0.643833,0.481389,0.063835,-0.199878,0.288357
Supplier8,0.012476,0.251014,-0.025427,-0.410723,0.640686,-0.525574,0.227811
Supplier7,-0.020235,0.000743,0.756044,0.600705,-0.381690,-0.043502,0.060847
Supplier5,0.498083,-0.112735,-0.156611,-0.004331,-0.142019,0.743812,0.027904
Supplier2,-0.081094,-0.194849,-0.493576,0.510884,0.415111,0.511211,-0.016227


In [2]:
# Install required packages (only run if not installed)
# !pip install gensim scikit-learn pandas openpyxl --quiet

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# Supplier Data
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']
suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "Strong in pricing and lead time but lacks security protocols. Focuses on sustainability.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing. Low-cost pricing with excellent lead time.",
    'Supplier4': "Very costly. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "Good security and compliance, decent delivery, quality-focused. Balanced approach with pricing.",
    'Supplier7': "Imbalanced approach with pricing. Good security. Bad lead time.",
    'Supplier8': "Excellent lead time and low pricing. Lacks sustainability credentials and weak compliance.",
    'Supplier9': "ISO certified, strong compliance, and sustainable processes. Moderate pricing and solid quality.",
    'Supplier10': "High-end quality assurance, strong security, but high cost and slower delivery times."
}


positive_words = {'good', 'great', 'excellent', 'strong', 'fast', 'high', 'balanced'}
negative_words = {'bad', 'poor', 'slow', 'weak', 'low', 'imbalanced', 'excessive', 'lacks', 'decreased'}

def get_sentiment_scale(word, full_text):
    for pos in positive_words:
        if f"{pos} {word}" in full_text:
            return 1.2
    for neg in negative_words:
        if f"{neg} {word}" in full_text:
            return -0.8
    return 0.9  # Neutral

def get_full_scores(extra_keywords=[]):
    keywords = nike_criteria + extra_keywords

    def extract_keywords(text):
        return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]

    tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]
    model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1, seed=42)

    results_weighted = {}
    for name, words in zip(suppliers.keys(), tokenized_data):
        word_freq = Counter(words)
        full_text = suppliers[name].lower()
        scores = []

        for crit in nike_criteria:
            if crit not in model.wv:
                scores.append(0.0)
                continue

            crit_vec = model.wv[crit].reshape(1, -1)
            weighted_vectors = []

            for w in words:
                if w in model.wv:
                    weight = get_sentiment_scale(w, full_text)
                    weighted_vectors.append(model.wv[w] * word_freq[w] * weight)

            if weighted_vectors:
                avg_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
                sim = cosine_similarity(avg_vec, crit_vec).flatten()[0]
            else:
                sim = 0.0

            scores.append(sim)

        results_weighted[name] = scores

    df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T

    # Apply Criteria Weights
    criteria_weights = {
        'quality': 0.8,
        'delivery': 0.25,
        'lead': 0.2,
        'pricing': 0.15,
        'security': 0.07,
        'sustainability': 0.03
    }

    df_scores["TotalMatchScore"] = sum(
        df_scores[crit] * criteria_weights[crit] for crit in nike_criteria
    )
    df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)
    return df_scores

# Generate and display the final output table
final_output_table = get_full_scores([])

# Display the table
print("\nFinal Supplier Evaluation Scores (Without Extra Keywords):\n")
print(final_output_table)

# Save results to CSV and Excel files
final_output_table.to_csv("Supplier_Scores_Without_Extra_Keywords.csv")
final_output_table.to_excel("Supplier_Scores_Without_Extra_Keywords.xlsx")

print("\nFiles have been saved as 'Supplier_Scores_Without_Extra_Keywords.csv' and 'Supplier_Scores_Without_Extra_Keywords.xlsx'.")



Final Supplier Evaluation Scores (Without Extra Keywords):

            delivery   quality  security   pricing      lead  sustainability  \
Supplier10  0.392359  0.575994  0.693973 -0.053491  0.086974        0.042987   
Supplier3   0.012672  0.523095  0.121510  0.410439  0.596850        0.054075   
Supplier9  -0.062308  0.676388  0.065786  0.717747 -0.199976       -0.124218   
Supplier1   0.667633  0.469271  0.067340 -0.095069  0.156865        0.421558   
Supplier6   0.392187  0.050218  0.656011  0.523422 -0.019495        0.065330   
Supplier4   0.083743 -0.040487  0.088926 -0.233128  1.000000        0.204832   
Supplier5   0.533218 -0.111151  0.088171 -0.091956  0.227386        0.778430   
Supplier8   0.145363  0.038043 -0.037652 -0.639361  0.733176       -0.310505   
Supplier7  -0.176872  0.079572  0.650847  0.618343 -0.501717        0.028923   
Supplier2   0.029216 -0.156503 -0.351183  0.425520  0.486857        0.528428   

            TotalMatchScore  
Supplier10         0.618124 

In [12]:
!pip install gensim scikit-learn pandas --quiet

In [31]:
!pip install gensim scikit-learn pandas --quiet
# Step 1: Setup
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from collections import Counter
# Step 2: Define Nike's criteria and supplier profiles
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']

suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "good in pricing and lead time also security protocols.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing.low-cost pricing with excellent lead time",
    'Supplier4': "low-cost pricing with frequent production delays. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "good security and compliance, decent delivery,  quality-focused.  Balanced approach with pricing, bad quality",
    'Supplier7': "imBalanced approach with pricing, . bad security, Lead time needs improvement."
}
# Step 3: Extract keyword-based tokens from supplier descriptions
keywords = nike_criteria + ['compliance', 'green', 'logistics','quality']
def extract_keywords(text):
    return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]

tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]
results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    scores = []
    for crit in nike_criteria:
        if crit in model.wv and words:
            crit_vec = model.wv[crit].reshape(1, -1)
            # Average all supplier vectors (weighted by frequency)
            weighted_vectors = [model.wv[w] * word_freq[w] for w in words if w in model.wv]
            avg_supplier_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_supplier_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0
        scores.append(sim)
    results_weighted[name] = scores

df_scores_weighted = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores_weighted["TotalMatchScore"] = df_scores_weighted.sum(axis=1)
df_scores_weighted = df_scores_weighted.sort_values("TotalMatchScore", ascending=False)
df_scores_weighted


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier7,0.087550,-0.182091,0.637896,0.663140,0.529313,0.000632,1.736440
Supplier2,0.087550,-0.182091,0.637896,0.663140,0.529313,0.000632,1.736440
Supplier6,0.345867,0.265865,0.585100,0.459324,0.037356,0.031150,1.724661
Supplier4,-0.006729,-0.122991,0.100608,0.765048,0.675812,0.020960,1.432708
Supplier1,0.564233,0.380140,-0.015038,-0.075538,-0.022333,0.596055,1.427517
Supplier5,0.724587,-0.279534,0.088986,-0.068655,0.094285,0.759707,1.319376
Supplier3,0.010285,0.397672,-0.042284,0.537024,0.235192,-0.115137,1.022752


In [32]:
# Fully corrected: use all matching words per criterion for cosine similarity (not just the word itself)

# Train Word2Vec again
model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1)

results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    full_text = suppliers[name].lower()
    scores = []

    for crit in nike_criteria:
        if crit not in model.wv:
            scores.append(0.0)
            continue

        crit_vec = model.wv[crit].reshape(1, -1)

        # Use all matching words for this supplier — not just exact 'crit'
        weighted_vectors = []
        for w in words:
            if w in model.wv:
                weight = get_sentiment_scale(w, full_text)
                weighted_vectors.append(model.wv[w] * word_freq[w] * weight)

        if weighted_vectors:
            avg_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0

        scores.append(sim)

    results_weighted[name] = scores

# Create DataFrame and calculate final score
df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores["TotalMatchScore"] = df_scores.sum(axis=1)
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)

df_scores


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier2,0.087550,-0.182091,0.637896,0.663140,0.529313,0.000632,1.736440
Supplier1,0.684001,0.297669,0.017433,-0.088667,0.003425,0.545627,1.459488
Supplier4,-0.006729,-0.122991,0.100608,0.765048,0.675812,0.020960,1.432708
Supplier6,0.443096,-0.505051,0.706590,0.395437,0.153207,0.167595,1.360874
Supplier5,0.631406,-0.276644,0.069668,-0.055512,0.083127,0.835988,1.288034
Supplier3,0.026965,0.357453,-0.039029,0.520747,0.362497,-0.107899,1.120734
Supplier7,-0.099937,-0.020481,-0.478723,0.603208,0.590083,0.034296,0.628446


In [36]:
# Step 1: Install necessary libraries (for Colab or local)
!pip install gensim scikit-learn pandas --quiet

# Step 2: Imports
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from collections import Counter

# Step 3: Define criteria and supplier data
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']

suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "Strong in pricing and lead time but lacks security protocols.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing. low-cost pricing with excellent lead time",
    'Supplier4': "Low-cost pricing with frequent production delays. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "Good security and compliance, decent delivery, quality-focused. Balanced approach with pricing. Bad quality.",
    'Supplier7': "Imbalanced approach with pricing. Bad security. Lead time needs improvement."
}

# Step 4: Extract keywords from text (tokens match criteria list)
keywords = nike_criteria + ['compliance', 'green', 'logistics']
def extract_keywords(text):
    return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]

tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]

# Step 5: Define sentiment-based scaling
positive_words = {'good', 'great', 'excellent', 'strong', 'fast', 'high'}
negative_words = {'bad', 'poor', 'slow', 'weak', 'low', 'imbalanced', 'excessive', 'lacks'}

def get_sentiment_scale(word, full_text):
    for pos in positive_words:
        if f"{pos} {word}" in full_text:
            return 1.2  # boost good context
    for neg in negative_words:
        if f"{neg} {word}" in full_text:
            return -0.8  # penalize bad context
    return 0.8  # neutral weight

# Step 6: Train Word2Vec model on supplier token data
model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1)

# Step 7: Score each supplier using sentiment and zeroing absent criteria
results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    full_text = suppliers[name].lower()
    scores = []
    for crit in nike_criteria:
        if crit not in words or crit not in model.wv:
            # If the criterion word is completely missing, score is zero
            scores.append(0.0)
            continue

        crit_vec = model.wv[crit].reshape(1, -1)
        weighted_vectors = []
        for w in words:
            if w == crit and w in model.wv:
                weight = get_sentiment_scale(w, full_text)
                weighted_vectors.append(model.wv[w] * word_freq[w] * weight)

        if weighted_vectors:
            avg_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0
        scores.append(sim)
    results_weighted[name] = scores

# Step 8: Create results table
df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores['TotalMatchScore'] = df_scores.sum(axis=1)
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)

# Step 9: Display final table
df_scores


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier3,0.0,1.0,0.0,1.0,1.0,0.0,3.0
Supplier1,1.0,1.0,0.0,0.0,0.0,1.0,3.0
Supplier5,1.0,0.0,0.0,0.0,0.0,1.0,2.0
Supplier6,1.0,-1.0,1.0,1.0,0.0,0.0,2.0
Supplier4,0.0,0.0,0.0,1.0,1.0,0.0,2.0
Supplier2,0.0,0.0,-1.0,1.0,1.0,0.0,1.0
Supplier7,0.0,0.0,-1.0,1.0,1.0,0.0,1.0


In [26]:
# Re-import required packages after reset
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from collections import Counter

# Supplier data
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']
suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "Strong in pricing and lead time but lacks security protocols.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing. low-cost pricing with excellent lead time",
    'Supplier4': "Low-cost pricing with frequent production delays. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "Good security and compliance, decent delivery, quality-focused. Balanced approach with pricing. Bad quality.",
    'Supplier7': "Imbalanced approach with pricing. Bad security. Lead time needs improvement."
}

# Keywords and tokenization
keywords = nike_criteria + ['compliance', 'green', 'logistics']
def extract_keywords(text):
    return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]
tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]

# Sentiment weights
positive_words = {'good', 'great', 'excellent', 'strong', 'fast', 'high', 'balanced'}
negative_words = {'bad', 'poor', 'slow', 'weak', 'low', 'imbalanced', 'excessive', 'lacks', 'decreased'}

def get_sentiment_scale(word, full_text):
    for pos in positive_words:
        if f"{pos} {word}" in full_text:
            return 1.2
    for neg in negative_words:
        if f"{neg} {word}" in full_text:
            return -0.8
    return 0.9

# Train Word2Vec
model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1)

# Score calculation
results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    full_text = suppliers[name].lower()
    scores = []
    for crit in nike_criteria:
        if crit not in model.wv:
            scores.append(0.0)
            continue
        crit_vec = model.wv[crit].reshape(1, -1)
        relevant_words = [w for w in words if w == crit and w in model.wv]
        if not relevant_words:
            scores.append(0.0)
            continue
        vectors = []
        for w in relevant_words:
            weight = get_sentiment_scale(w, full_text)
            vectors.append(model.wv[w] * word_freq[w] * weight)
        if vectors:
            avg_vec = np.mean(vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0
        scores.append(sim)
    results_weighted[name] = scores

# Create result dataframe
df_scores = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores["TotalMatchScore"] = df_scores.sum(axis=1)
df_scores = df_scores.sort_values("TotalMatchScore", ascending=False)

#import ace_tools as tools; tools.display_dataframe_to_user(name="Supplier Evaluation Scores", dataframe=df_scores)
df_scores


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier1,1.0,1.0,0.0,0.0,0.0,1.0,3.0
Supplier3,0.0,1.0,0.0,1.0,1.0,0.0,3.0
Supplier5,1.0,0.0,0.0,0.0,0.0,1.0,2.0
Supplier6,1.0,-1.0,1.0,1.0,0.0,0.0,2.0
Supplier4,0.0,0.0,0.0,1.0,1.0,0.0,2.0
Supplier2,0.0,0.0,-1.0,1.0,1.0,0.0,1.0
Supplier7,0.0,0.0,-1.0,1.0,1.0,0.0,1.0


In [8]:
# Step 2: Define Nike's criteria and supplier profiles
nike_criteria = ['delivery', 'quality', 'security', 'pricing', 'lead', 'sustainability']

suppliers = {
    'Supplier1': "Provides fast delivery and great product quality. Focuses on sustainability.",
    'Supplier2': "Strong in pricing and lead time but lacks security protocols.",
    'Supplier3': "Delivers secure logistics, maintains product quality, and emphasizes green manufacturing.low-cost pricing with excellent lead time",
    'Supplier4': "low-cost pricing with frequent production delays. Product failures decreased. Lead time is excessive.",
    'Supplier5': "High sustainability commitment, ISO certified, but slower delivery.",
    'Supplier6': "bad in security and compliance, decent delivery, not quality-focused. not Balanced approach with pricing, bad quality",
    'Supplier7': "imBalanced approach with pricing, quality, and sustainability. bad Lead time needs improvement."
}

In [9]:
# Step 3: Extract keyword-based tokens from supplier descriptions
keywords = nike_criteria + ['compliance', 'green', 'logistics']
def extract_keywords(text):
    return [word.strip(".,").lower() for word in text.split() if word.strip(".,").lower() in keywords]

tokenized_data = [extract_keywords(desc) for desc in suppliers.values()]

In [10]:
# Step 4: Train Word2Vec model
model = Word2Vec(sentences=tokenized_data, vector_size=50, window=2, min_count=1, workers=1)

In [11]:
results_weighted = {}
for name, words in zip(suppliers.keys(), tokenized_data):
    word_freq = Counter(words)
    scores = []
    for crit in nike_criteria:
        if crit in model.wv and words:
            crit_vec = model.wv[crit].reshape(1, -1)
            # Average all supplier vectors (weighted by frequency)
            weighted_vectors = [model.wv[w] * word_freq[w] for w in words if w in model.wv]
            avg_supplier_vec = np.mean(weighted_vectors, axis=0).reshape(1, -1)
            sim = cosine_similarity(avg_supplier_vec, crit_vec).flatten()[0]
        else:
            sim = 0.0
        scores.append(sim)
    results_weighted[name] = scores

df_scores_weighted = pd.DataFrame(results_weighted, index=nike_criteria).T
df_scores_weighted["TotalMatchScore"] = df_scores_weighted.sum(axis=1)
df_scores_weighted = df_scores_weighted.sort_values("TotalMatchScore", ascending=False)
df_scores_weighted


,delivery,quality,security,pricing,lead,sustainability,TotalMatchScore
Supplier7,-0.050603,0.552832,0.215422,0.511968,0.431870,0.431026,2.092516
Supplier1,0.582695,0.620724,-0.305477,0.015733,0.080311,0.661549,1.655535
Supplier6,0.437308,0.474635,0.246708,0.493708,-0.009277,-0.049693,1.593388
Supplier4,0.020960,0.100608,-0.122991,0.765048,0.675812,-0.006729,1.432708
Supplier2,-0.108858,-0.006588,0.500757,0.658481,0.483502,-0.131127,1.396166
Supplier3,-0.029851,0.485549,-0.133609,0.574941,0.305492,0.176341,1.378864
Supplier5,0.759735,0.088980,-0.279534,-0.068651,0.094281,0.724557,1.319369
